## Pandas performance
Comparing runtime performance of Pandas apply vs Pandas vector vs Numpy vs Polars

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import time

import timeit

In [2]:
def best_ms(fn, n=10):
    result = fn()                                          # one untimed run → the output
    ms = min(timeit.repeat(fn, number=1, repeat=n)) * 1000
    return ms, result

In [3]:
results = []

## Task 1 - Late Delivery Flag|

In [4]:
task = "late_delivery_flag"
print(f"Task: {task}")

# Pandas Apply Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
pandas_apply_ms, pandas_apply_result = best_ms(
    lambda: orders.apply(
        lambda x: x["delivery_min"] > 15 if x["city"] in ["Delhi", "Mumbai", "Bengaluru"] else x["delivery_min"] > 20,
        axis=1,
    )
)
ref = np.asarray(pandas_apply_result)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
pandas_vector_ms, pandas_vector_result = best_ms(
    lambda: orders["delivery_min"] > orders["city"].map({"Delhi": 15, "Mumbai": 15, "Bengaluru": 15}).fillna(20)
)
assert (np.asarray(pandas_vector_result) == ref).all()
print(f"Time taken by pandas_vector: {pandas_vector_ms:.2f} ms")

# Numpy Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
numpy_ms, numpy_result = best_ms(
    lambda: np.where(orders["city"].isin(["Delhi", "Mumbai", "Bengaluru"]), orders["delivery_min"] > 15, orders["delivery_min"] > 20)
)
assert (np.asarray(numpy_result) == ref).all()
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars
orders = pl.read_csv("../data/orders.csv", try_parse_dates=True)
polars_ms, polars_result = best_ms(
    lambda: orders.with_columns(
        pl.when(pl.col("city").is_in(["Delhi", "Mumbai", "Bengaluru"]))
        .then(pl.col("delivery_min") > 15)
        .otherwise(pl.col("delivery_min") > 20)
        .alias("is_late")
    )
)
assert (np.asarray(polars_result["is_late"]) == ref).all()
print(f"Time taken by polars: {polars_ms:.2f} ms")

results.append([task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms])


Task: late_delivery_flag


Time taken by pandas_apply: 309.25 ms
Time taken by pandas_vector: 2.62 ms


Time taken by numpy: 1.98 ms
Time taken by polars: 0.50 ms


## Task 2 - Basket tier

In [5]:
task = "Basket tier"
print(f"Task: {task}")

# Pandas Apply Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

assert pd.api.types.is_integer_dtype(orders["items"]) and (orders["items"] >= 1).all(), \
    "basket tiers assume integer items >= 1"

pandas_apply_ms, pandas_apply_result = best_ms(
    lambda: orders.apply(
        lambda x:
        "mini" if x["items"] <= 2
        else "regular" if x["items"] <= 5
        else "large" if x["items"] <= 9
        else "stock_up",
        axis = 1
    )
)
ref = np.asarray(pandas_apply_result)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

assert pd.api.types.is_integer_dtype(orders["items"]) and (orders["items"] >= 1).all(), \
    "basket tiers assume integer items >= 1"

pandas_vector_ms, pandas_vector_result = best_ms(
    lambda: pd.cut(
        orders["items"],
        bins = [0, 2, 5, 9, np.inf],
        labels = ["mini", "regular", "large", "stock_up"]
    )
)

assert (np.asarray(pandas_vector_result) == ref).all()
print(f"Time taken by pandas_vector: {pandas_vector_ms:.2f} ms")

# Numpy Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

assert pd.api.types.is_integer_dtype(orders["items"]) and (orders["items"] >= 1).all(), \
    "basket tiers assume integer items >= 1"

numpy_ms, numpy_result = best_ms(
    lambda: np.select(
        [orders["items"] <= 2, orders["items"] <= 5, orders["items"] <= 9],
        ["mini", "regular", "large"],
        default = "stock_up"
    )
)

assert (np.asarray(numpy_result) == ref).all()
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars Function
orders = pl.read_csv("../data/orders.csv", try_parse_dates=True)
polars_ms, polars_result = best_ms(
    lambda: orders.with_columns(
        pl.when(pl.col("items") <= 2).then(pl.lit("mini"))
        .when(pl.col("items") <= 5).then(pl.lit("regular"))
        .when(pl.col("items") <= 9).then(pl.lit("large"))
        .otherwise(pl.lit("stock_up"))
        .alias("basket_tier")
    )
)
assert (np.asarray(polars_result["basket_tier"]) == ref).all()
print(f"Time taken by polars: {polars_ms:.2f} ms")

results.append([task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms])


Task: Basket tier


Time taken by pandas_apply: 319.82 ms
Time taken by pandas_vector: 1.26 ms


Time taken by numpy: 1.47 ms
Time taken by polars: 0.55 ms


## Task 3 - Days since previous order

In [6]:
task = "days_since_prev_order"
print(f"Task: {task}")

# Pandas Apply Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

def pandas_apply_days_since_prev(orders):
    orders = orders.sort_values(by = ["user_id", "order_ts"], ascending = [True, True])
    return (orders["order_ts"] - orders.groupby("user_id", group_keys=False)["order_ts"].apply(lambda x: x.shift(1)))/pd.Timedelta(days=1)

pandas_apply_ms, pandas_apply_result = best_ms(lambda: pandas_apply_days_since_prev(orders))
ref = np.asarray(pandas_apply_result)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

def pandas_vector_days_since_prev(orders):
    orders = orders.sort_values(by = ["user_id", "order_ts"], ascending = [True, True])
    return orders["order_ts"].groupby(orders["user_id"]).diff() / pd.Timedelta(days=1)

pandas_vector_ms, pandas_vector_result = best_ms(lambda: pandas_vector_days_since_prev(orders))
assert np.allclose(np.asarray(pandas_vector_result), ref, equal_nan = True)
print(f"Time taken by pandas_vector: {pandas_vector_ms:.2f} ms")

# Numpy Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

def numpy_days_since_prev(orders):
    orders = orders.sort_values(["user_id", "order_ts"])          # sort is now part of the timed work
    return np.where(
        np.concatenate(([np.nan], orders["user_id"][:-1])) == orders["user_id"],
        (orders["order_ts"] - np.concatenate((np.array(["NaT"], dtype=orders["order_ts"].to_numpy().dtype), orders["order_ts"][:-1])))/pd.Timedelta(days=1),
        np.nan
    )

numpy_ms, numpy_result = best_ms(lambda: numpy_days_since_prev(orders))
assert np.allclose(np.asarray(numpy_result), ref, equal_nan = True)
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars Function
orders = pl.read_csv("../data/orders.csv", try_parse_dates=True)
polars_ms, polars_result = best_ms(
    lambda: orders.sort(["user_id", "order_ts"])
    .with_columns(
        (pl.col("order_ts").diff().over("user_id") / pl.duration(days=1))
        .alias("days_since_prev")
    )
)
assert np.allclose(np.asarray(polars_result["days_since_prev"]), ref, equal_nan = True)
print(f"Time taken by polars: {polars_ms:.2f} ms")

results.append([task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms])


Task: days_since_prev_order


Time taken by pandas_apply: 650.47 ms


Time taken by pandas_vector: 15.40 ms


Time taken by numpy: 13.01 ms


Time taken by polars: 28.87 ms


## Where does task 3's time go?

In [7]:
orders_pd = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
orders_pl = pl.read_csv("../data/orders.csv", try_parse_dates=True)

pd_sort_ms, _ = best_ms(lambda: orders_pd.sort_values(["user_id", "order_ts"]))
pl_sort_ms, _ = best_ms(lambda: orders_pl.sort(["user_id", "order_ts"]))
pl_over_ms, _ = best_ms(lambda: orders_pl.with_columns(pl.col("order_ts").diff().over("user_id")))
print(pd_sort_ms, pl_sort_ms, pl_over_ms)

11.558900005184114 2.192899992223829 27.28079998632893


## 10× scale

In [8]:
task = "days_since_prev_order_10x"
print(f"Task: {task}")

# 10 copies of the orders, each with its own block of user_ids
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
max_user = orders["user_id"].max()
orders_10x = pd.concat(
    [orders.assign(user_id=orders["user_id"] + k * max_user) for k in range(10)],
    ignore_index=True,
)
print(f"Rows: {len(orders_10x):,}")

# Pandas Apply Function (fewer repeats, each run takes seconds at this size)
pandas_apply_ms, pandas_apply_result = best_ms(lambda: pandas_apply_days_since_prev(orders_10x), n=3)
ref = np.asarray(pandas_apply_result)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
pandas_vector_ms, pandas_vector_result = best_ms(lambda: pandas_vector_days_since_prev(orders_10x))
assert np.allclose(np.asarray(pandas_vector_result), ref, equal_nan = True)
print(f"Time taken by pandas_vector: {pandas_vector_ms:.2f} ms")

# Numpy Function
numpy_ms, numpy_result = best_ms(lambda: numpy_days_since_prev(orders_10x))
assert np.allclose(np.asarray(numpy_result), ref, equal_nan = True)
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars Function
orders_pl = pl.read_csv("../data/orders.csv", try_parse_dates=True)
orders_10x_pl = pl.concat(
    [orders_pl.with_columns(pl.col("user_id") + k * max_user) for k in range(10)]
)
polars_ms, polars_result = best_ms(
    lambda: orders_10x_pl.sort(["user_id", "order_ts"])
    .with_columns(
        (pl.col("order_ts").diff().over("user_id") / pl.duration(days=1))
        .alias("days_since_prev")
    )
)
assert np.allclose(np.asarray(polars_result["days_since_prev"]), ref, equal_nan = True)
print(f"Time taken by polars: {polars_ms:.2f} ms")

results_10x_df = pd.DataFrame(
    [[task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms]],
    columns = ["task", "pandas_apply_ms", "pandas_vector_ms", "numpy_ms", "polars_ms"]
)

results_10x_df["vector_speedup"] = results_10x_df["pandas_apply_ms"]/results_10x_df["pandas_vector_ms"]
results_10x_df["numpy_speedup"] = results_10x_df["pandas_apply_ms"]/results_10x_df["numpy_ms"]
results_10x_df["polars_speedup"] = results_10x_df["pandas_apply_ms"]/results_10x_df["polars_ms"]

results_10x_df

Task: days_since_prev_order_10x
Rows: 960,200


Time taken by pandas_apply: 6897.65 ms


Time taken by pandas_vector: 190.32 ms


Time taken by numpy: 165.92 ms


Time taken by polars: 289.05 ms


,task,pandas_apply_ms,pandas_vector_ms,numpy_ms,polars_ms,vector_speedup,numpy_speedup,polars_speedup
0,days_since_prev_order_10x,6897.6539,190.3176,165.9238,289.052,36.242859,41.571215,23.863021


## Result and Conclusion

In [9]:
results_df = pd.DataFrame(
    results,
    columns = ["task", "pandas_apply_ms", "pandas_vector_ms", "numpy_ms", "polars_ms"]
)

results_df["vector_speedup"] = results_df["pandas_apply_ms"]/results_df["pandas_vector_ms"]
results_df["numpy_speedup"] = results_df["pandas_apply_ms"]/results_df["numpy_ms"]
results_df["polars_speedup"] = results_df["pandas_apply_ms"]/results_df["polars_ms"]

results_df

,task,pandas_apply_ms,pandas_vector_ms,numpy_ms,polars_ms,vector_speedup,numpy_speedup,polars_speedup
0,late_delivery_flag,309.2503,2.6188,1.9826,0.5017,118.088552,155.982196,616.404814
1,Basket tier,319.8194,1.2619,1.4657,0.5546,253.442744,218.202499,576.666784
2,days_since_prev_order,650.4744,15.3983,13.0084,28.8656,42.243261,50.004182,22.534588


1. Replacing `apply` with vectorized code was 40–250× faster in every task.
2. Polars was fastest on the row-wise tasks (tasks 1 and 2), about 2–5× faster than pandas or NumPy.
3. Every task 3 version sorts inside its timer, so outputs are compared end to end.
4. In task 3, NumPy was ~15% faster than pandas `groupby().diff()` (14.6 vs 17.5 ms), but the pandas sort alone takes ~12 ms, so both are dominated by the sort. NumPy also relies on rows being sorted by user, so groupby is the safer default.
5. Polars was the slowest vectorized version in task 3 (30 ms), and the sort isn't why: its sort takes ~2 ms, while `diff().over("user_id")` takes ~28 ms.
6. With 10× the orders (~960k rows), every task 3 version took about 10× as long and the ranking didn't change: NumPy 162 ms, pandas 188 ms, Polars 287 ms, `apply` 6.7 s.